In [10]:
import pandas as pd
import os
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score, classification_report, roc_auc_score,
    precision_score, recall_score, f1_score
)


In [11]:
# Load dataset
file_path = "../data/processed/dataset_final.csv"

try:
    df = pd.read_csv(file_path)
    print("✅ Data loaded successfully!")
    print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")
except FileNotFoundError:
    raise FileNotFoundError(f"❌ Could not find file at {file_path}")


✅ Data loaded successfully!
Rows: 1200, Columns: 41


In [12]:
cols_to_drop = [
    'transaction_id', 'timestamp', 'customer_id',
    'transaction_description', 'merchant_name',
    'customer_support_note', 'risk_category'
]

df_clean = df.drop(columns=cols_to_drop, errors='ignore')


In [13]:
# One-hot encode categorical variables
df_encoded = pd.get_dummies(df_clean, drop_first=True)

# Split features and target
X = df_encoded.drop('is_fraud', axis=1)
y = df_encoded['is_fraud']

# Train-test split (stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Feature scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training Shape: {X_train_scaled.shape}")
print(f"Testing Shape: {X_test_scaled.shape}")


Training Shape: (960, 40)
Testing Shape: (240, 40)


In [14]:
mlp_baseline = MLPClassifier(
    hidden_layer_sizes=(50,),
    max_iter=300,
    random_state=42
)

mlp_baseline.fit(X_train_scaled, y_train)

y_pred_base = mlp_baseline.predict(X_test_scaled)
y_prob_base = mlp_baseline.predict_proba(X_test_scaled)[:, 1]

print("--- Model E Baseline Performance ---")
print(f"Accuracy: {accuracy_score(y_test, y_pred_base):.4f}")
print(f"ROC-AUC: {roc_auc_score(y_test, y_prob_base):.4f}")
print(classification_report(y_test, y_pred_base))


--- Model E Baseline Performance ---
Accuracy: 0.8083
ROC-AUC: 0.6555
              precision    recall  f1-score   support

           0       0.84      0.93      0.88       188
           1       0.59      0.38      0.47        52

    accuracy                           0.81       240
   macro avg       0.72      0.66      0.67       240
weighted avg       0.79      0.81      0.79       240



c:\Users\USER\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(


In [15]:
param_grid = {
    'hidden_layer_sizes': [(50,), (100,), (50, 50)],
    'alpha': [0.0001, 0.001],
    'learning_rate_init': [0.001, 0.01]
}

grid_search = GridSearchCV(
    estimator=MLPClassifier(max_iter=500, random_state=42),
    param_grid=param_grid,
    cv=3,
    scoring='roc_auc',
    n_jobs=1,
    verbose=1
)

print("Starting Model E Hyperparameter Tuning...")
grid_search.fit(X_train_scaled, y_train)

best_mlp = grid_search.best_estimator_

print("✅ Tuning Complete")
print(f"Best Parameters: {grid_search.best_params_}")


Starting Model E Hyperparameter Tuning...
Fitting 3 folds for each of 12 candidates, totalling 36 fits


c:\Users\USER\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\USER\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\USER\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\USER\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimiza

✅ Tuning Complete
Best Parameters: {'alpha': 0.0001, 'hidden_layer_sizes': (50, 50), 'learning_rate_init': 0.01}


In [16]:
y_pred_tuned = best_mlp.predict(X_test_scaled)
y_prob_tuned = best_mlp.predict_proba(X_test_scaled)[:, 1]

acc_tuned = accuracy_score(y_test, y_pred_tuned)
auc_tuned = roc_auc_score(y_test, y_prob_tuned)

print("--- Model E Tuned Performance ---")
print(f"Accuracy: {acc_tuned:.4f}")
print(f"ROC-AUC: {auc_tuned:.4f}")
print(classification_report(y_test, y_pred_tuned))


--- Model E Tuned Performance ---
Accuracy: 0.7500
ROC-AUC: 0.6842
              precision    recall  f1-score   support

           0       0.83      0.85      0.84       188
           1       0.42      0.38      0.40        52

    accuracy                           0.75       240
   macro avg       0.62      0.62      0.62       240
weighted avg       0.74      0.75      0.75       240

